# 03 · Gold · `fato_segmento_mensal`
### V-Energy Grid 360 — Arquitetura Medalhão

Pedido de Dev: a `kpis_mensal` tem uma linha por mês e não filtra por município, bairro ou classe; a `receita_decomposicao` só compara os dois períodos fixos do diagnóstico. Esta tabela agrega consumo, receita e faturas **por segmento e mês**: o dashboard filtra direto nela, e a API calcula a decomposição preço × volume para **qualquer par de períodos**. Grava `v_energy.gold.fato_segmento_mensal`, seguindo o **contrato da Gold (tabela A3, versão v4)**, com as 16 colunas do contrato.

**Responde a:** P3 (indicadores por segmento), P2 (decomposição em qualquer período), P12 (filtros), P10 ("qual bairro mais perdeu volume residencial no último ano?", "quanto a receita comercial cresceu em Jaboatão?") e P11 (base dos insights).

**Grão:** 1 linha = 1 mês × 1 município × 1 bairro × 1 classe, **só as combinações que existem**: o segmento entra no mês em que tem pelo menos uma UC ligada.

**Fontes (Gold):** `fato_consumo_faturamento` (consumo, receita, faturas, GD no mês e trocas) e `dim_uc` (UCs ligadas até o fim de cada mês).

| Coluna | Regra |
|---|---|
| `qtd_ucs_cadastradas` | UCs do segmento com `data_ligacao` até o último dia do mês (`dim_uc`) |
| `qtd_ucs_faturadas` | UCs distintas do segmento com fatura no mês |
| `qtd_ucs_com_gd`, `consumo_kwh_ucs_com_gd` | linhas com `possui_gd_no_mes = true` (a regra "GD no mês" das outras tabelas) |
| `consumo_kwh`, `receita_energia`, `receita_total` | somas de `consumo_kwh`, `valor_energia` e `valor_total` |
| `qtd_faturas`, `qtd_faturas_em_aberto`, `valor_em_aberto` | faturas **desta competência**, e as dela ainda em aberto |
| `qtd_faturas_bandeira_nao_verde` | faturas com bandeira diferente de `Verde` |
| `qtd_trocas_medidor` | linhas com `troca_medidor = true` (6.299 no total) |

**Regra para a API (contrato):** a decomposição de dois períodos quaisquer soma `consumo_kwh` e `receita_energia` de cada período e aplica as fórmulas da A4 (`efeito_volume = (V1 − V0) × P0`; `efeito_preco = variação − efeito_volume`; volume zero: tudo é efeito-volume). **Nunca somar efeitos de segmentos** (efeito composição; ver a `receita_decomposicao`). A seção 9 mostra a função pronta.

**Decisões deste notebook:**
- **Segmento com UC ligada, mas sem fatura no mês**, entra com as somas em **zero** (a UC é faturada a partir do mês seguinte ao da ligação). Assim, `qtd_ucs_cadastradas` não perde nenhuma UC.
- **Não há colunas de preço nem de percentual:** a tabela só guarda somas e contagens, que são **aditivas**. Preço, inadimplência e % de bandeira são calculados depois de somar (`receita ÷ volume`, `valor_em_aberto ÷ receita_total`), em qualquer recorte.
- **Sem tabela temporária (`_stg`):** a agregação lê a fato uma vez; as checagens leem a tabela agregada (algumas dezenas de milhares de linhas).

**Pré-requisito:** `gold.fato_consumo_faturamento` e `gold.dim_uc` gravadas. Para as conferências, a `kpis_mensal` e a `receita_decomposicao` (se existirem no workspace).

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from decimal import Decimal
from pyspark.sql import functions as F, Row, Window
from datetime import datetime

tabela_fato = f"{gold_schema}.fato_consumo_faturamento"
tabela_dim = f"{gold_schema}.dim_uc"
tabela_kpis = f"{gold_schema}.kpis_mensal"             # só para conferência (seção 9)
tabela_rec = f"{gold_schema}.receita_decomposicao"     # só para conferência (seção 9)
tabela_gold = f"{gold_schema}.fato_segmento_mensal"
tabela_dq_log = f"{silver_schema}.dq_log"              # mesmo log das Silvers e das outras Golds

MES_INI, MES_FIM = "2022-07", "2026-06"
P0_INI, P0_FIM = "2022-07", "2023-06"                  # primeiro ano
P1_INI, P1_FIM = "2025-07", "2026-06"                  # últimos 12 meses
PA_INI, PA_FIM = "2024-07", "2025-06"                  # 12 meses anteriores (o "último ano" do Henrique)
CHAVE = ["mes_referencia", "municipio", "bairro", "classe"]

def n_br(n):
    return f"{n:,}".replace(",", ".")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"origem:  {tabela_fato}, {tabela_dim}")
print(f"destino: {tabela_gold}")

## 2. Funções de checagem de qualidade

Mesmas funções das Silvers e das outras Golds, com as mesmas colunas no log. Os resultados vão para `silver.dq_log` (sempre), e a tabela só é gravada se **todas** as checagens passarem.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=failed, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=dupes, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def dq_check_valor(table_name: str, check_name: str, total_rows: int, obtido, esperado):
    """Compara um valor calculado com o esperado (soma que precisa bater com a fato ou com o diagnóstico)."""
    passed = obtido == esperado
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total_rows,
                          failed_rows=0 if passed else 1, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | obtido {obtido} (esperado {esperado})")

## 3. Leitura das fontes e conferência das colunas

Contagens esperadas: **2.152.336** linhas na fato e **45.000** UCs.

**Os atributos da UC são os mesmos nas duas tabelas:** a fato copia `classe`, `bairro` e `municipio` do cadastro, e o teste de conexão da `fato_consumo_faturamento` deu 0 diferenças com a `dim_uc`. O notebook confere de novo, porque a tabela junta as duas pelo segmento.

In [0]:
fato = spark.table(tabela_fato)
dim = spark.table(tabela_dim)

exigidas = {
    tabela_fato: ["id_uc", "mes_referencia", "municipio", "bairro", "classe", "possui_gd_no_mes", "consumo_kwh",
                  "valor_energia", "valor_total", "em_aberto", "bandeira_tarifaria", "troca_medidor"],
    tabela_dim: ["id_uc", "municipio", "bairro", "classe", "data_ligacao"],
}
for tabela, colunas in exigidas.items():
    faltando = [c for c in colunas if c not in spark.table(tabela).columns]
    if faltando:
        raise Exception(f"{tabela} não tem as colunas {faltando} (DESCRIBE {tabela}).")

n_fato, n_dim = fato.count(), dim.count()
print(f"fato: {n_br(n_fato)} | UCs: {n_br(n_dim)}")   # esperado: 2.152.336 | 45.000

dif = (fato.select("id_uc", "municipio", "bairro", "classe").distinct()
       .join(dim.select("id_uc", F.col("municipio").alias("m"), F.col("bairro").alias("b"), F.col("classe").alias("c")), "id_uc", "left")
       .filter(~(F.col("municipio").eqNullSafe(F.col("m")) & F.col("bairro").eqNullSafe(F.col("b")) & F.col("classe").eqNullSafe(F.col("c")))))
print("UCs com segmento diferente entre a fato e a dim_uc:", dif.count())   # esperado: 0
print("segmentos (município x bairro x classe) na dim_uc:", n_br(dim.select("municipio", "bairro", "classe").distinct().count()))

## 4. UCs cadastradas por segmento e mês

Junção por data: para cada mês, as UCs com `data_ligacao` até o último dia do mês, contadas por segmento. Só entram os pares segmento × mês com pelo menos uma UC ligada.

**Conferência:** 45.000 UCs somadas em `2026-06`.

In [0]:
meses = (spark.sql(f"SELECT explode(sequence(to_date('{MES_INI}-01'), to_date('{MES_FIM}-01'), interval 1 month)) AS ini")
         .select(F.date_format("ini", "yyyy-MM").alias("mes_referencia"), F.last_day("ini").alias("fim_mes")))

cad = (dim.select("id_uc", "municipio", "bairro", "classe", "data_ligacao")
       .join(F.broadcast(meses), F.col("data_ligacao") <= F.col("fim_mes"))
       .groupBy(*CHAVE).agg(F.count("*").alias("qtd_ucs_cadastradas")))

print("pares segmento x mês com UC ligada:", n_br(cad.count()))
print("UCs cadastradas em 2026-06:", n_br(cad.filter(f"mes_referencia = '{MES_FIM}'").agg(F.sum("qtd_ucs_cadastradas")).first()[0]))   # esperado: 45.000

## 5. Consumo, receita e faturas por segmento e mês

Uma passada pela fato, com somas e contagens condicionais. Tudo é **aditivo**: pode ser somado entre segmentos e entre meses.

**Conferência:** a soma das faturas é 2.152.336.

In [0]:
fat = fato.groupBy(*CHAVE).agg(
    F.countDistinct("id_uc").alias("qtd_ucs_faturadas"),
    F.countDistinct(F.when(F.col("possui_gd_no_mes"), F.col("id_uc"))).alias("qtd_ucs_com_gd"),
    F.sum("consumo_kwh").alias("consumo_kwh"),
    F.coalesce(F.sum(F.when(F.col("possui_gd_no_mes"), F.col("consumo_kwh"))), F.lit(0)).alias("consumo_kwh_ucs_com_gd"),
    F.sum("valor_energia").alias("receita_energia"),
    F.sum("valor_total").alias("receita_total"),
    F.count("*").alias("qtd_faturas"),
    F.sum(F.col("em_aberto").cast("int")).alias("qtd_faturas_em_aberto"),
    F.coalesce(F.sum(F.when(F.col("em_aberto"), F.col("valor_total"))), F.lit(0)).alias("valor_em_aberto"),
    F.sum(F.when(F.col("bandeira_tarifaria") != "Verde", 1).otherwise(0)).alias("qtd_faturas_bandeira_nao_verde"),
    F.sum(F.col("troca_medidor").cast("int")).alias("qtd_trocas_medidor"),
)
print("pares segmento x mês com fatura:", n_br(fat.count()))
print("faturas:", n_br(fat.agg(F.sum("qtd_faturas")).first()[0]))   # esperado: 2.152.336

# Algum segmento x mês com fatura e sem UC ligada? (esperado: 0; a UC é faturada depois da ligação)
print("pares com fatura e sem UC ligada:", fat.join(cad, CHAVE, "left_anti").count())

## 6. Montagem da `fato_segmento_mensal`

**Decisão:** partir dos pares com UC ligada (seção 4) e trazer as somas por `left join`. Par sem fatura no mês fica com **zero** nas somas e contagens. Se algum par tivesse fatura sem UC ligada, ele entraria pelo `full join` e a checagem da seção 7 acusaria (`qtd_ucs_cadastradas` nula).

**Ordem das colunas:** a do contrato (16 colunas).

In [0]:
somas = ["qtd_ucs_faturadas", "qtd_ucs_com_gd", "consumo_kwh", "consumo_kwh_ucs_com_gd", "receita_energia", "receita_total",
         "qtd_faturas", "qtd_faturas_em_aberto", "valor_em_aberto", "qtd_faturas_bandeira_nao_verde", "qtd_trocas_medidor"]

seg = cad.join(fat, CHAVE, "full").fillna(0, subset=somas)

df_seg = seg.select(
    "mes_referencia", "municipio", "bairro", "classe",
    F.col("qtd_ucs_cadastradas").cast("int").alias("qtd_ucs_cadastradas"),
    F.col("qtd_ucs_faturadas").cast("int").alias("qtd_ucs_faturadas"),
    F.col("qtd_ucs_com_gd").cast("int").alias("qtd_ucs_com_gd"),
    F.col("consumo_kwh").cast("decimal(18,2)").alias("consumo_kwh"),
    F.col("consumo_kwh_ucs_com_gd").cast("decimal(18,2)").alias("consumo_kwh_ucs_com_gd"),
    F.col("receita_energia").cast("decimal(18,2)").alias("receita_energia"),
    F.col("receita_total").cast("decimal(18,2)").alias("receita_total"),
    F.col("qtd_faturas").cast("int").alias("qtd_faturas"),
    F.col("qtd_faturas_em_aberto").cast("int").alias("qtd_faturas_em_aberto"),
    F.col("valor_em_aberto").cast("decimal(18,2)").alias("valor_em_aberto"),
    F.col("qtd_faturas_bandeira_nao_verde").cast("int").alias("qtd_faturas_bandeira_nao_verde"),
    F.col("qtd_trocas_medidor").cast("int").alias("qtd_trocas_medidor"),
)
print("colunas:", len(df_seg.columns))   # esperado: 16
print("linhas:", n_br(df_seg.count()))
df_seg.printSchema()

## 7. Checagens de qualidade

| Checagem | Protege |
|---|---|
| Chave `(mês, município, bairro, classe)` única; 48 meses; colunas não nulas | o grão |
| `classe` nas 5; 297 pares município + bairro | domínios (os do case) |
| `qtd_ucs_cadastradas` > 0 e nunca diminui no segmento; 45.000 em `2026-06` | o denominador |
| faturadas <= cadastradas; GD <= faturadas; faturas = UCs faturadas (1 por UC e mês) | coerência das contagens |
| consumo com GD <= consumo; em aberto <= faturas; valor em aberto <= receita total; não verdes <= faturas; receita total >= receita de energia | coerência das somas |
| **Somas iguais às da fato:** consumo, receitas, faturas, em aberto, valor em aberto, não verdes, trocas, linhas com GD | nada se perdeu nem duplicou |
| **Volume e receita dos dois períodos = os do diagnóstico** | reproduz os números do diretor |
| **3.510 UCs com GD em `2026-06`** e **6.299 trocas** | GD e trocas |

Os valores esperados valem para esta extração; numa reexecução com dados novos, precisam ser atualizados.

In [0]:
t = "gold.fato_segmento_mensal"
dq_results.clear()

CLASSES = ["RESIDENCIAL", "RURAL", "COMERCIAL", "PODER PUBLICO", "INDUSTRIAL"]
def ok(cond):
    return F.coalesce(cond, F.lit(False))   # nulo também falha

g = df_seg
n_g = g.count()

# O grão
dq_check_unique(t, "(mes, municipio, bairro, classe) único", g, CHAVE)
dq_check_valor(t, "meses distintos = 48", n_g, g.select("mes_referencia").distinct().count(), 48)
for coluna in df_seg.columns:
    dq_check(t, f"{coluna} não nulo", g, F.col(coluna).isNotNull())
dq_check(t, "classe nas 5 classes", g, ok(F.col("classe").isin(*CLASSES)))
dq_check_valor(t, "pares município + bairro = 297", n_g, g.select("municipio", "bairro").distinct().count(), 297)

# O denominador
dq_check(t, "qtd_ucs_cadastradas > 0", g, ok(F.col("qtd_ucs_cadastradas") > 0))
w_seg = Window.partitionBy("municipio", "bairro", "classe").orderBy("mes_referencia")
dq_check(t, "qtd_ucs_cadastradas não diminui no segmento",
         g.withColumn("ant", F.lag("qtd_ucs_cadastradas").over(w_seg)),
         F.col("ant").isNull() | (F.col("qtd_ucs_cadastradas") >= F.col("ant")))
dq_check_valor(t, "UCs cadastradas em 2026-06 = 45.000", n_g,
               g.filter(f"mes_referencia = '{MES_FIM}'").agg(F.sum("qtd_ucs_cadastradas")).first()[0], 45_000)

# Coerência das contagens e das somas
dq_check(t, "faturadas <= cadastradas", g, F.col("qtd_ucs_faturadas") <= F.col("qtd_ucs_cadastradas"))
dq_check(t, "UCs com GD <= faturadas", g, F.col("qtd_ucs_com_gd") <= F.col("qtd_ucs_faturadas"))
dq_check(t, "faturas = UCs faturadas (1 por UC e mês)", g, F.col("qtd_faturas") == F.col("qtd_ucs_faturadas"))
dq_check(t, "consumo com GD <= consumo; tudo >= 0", g,
         (F.col("consumo_kwh_ucs_com_gd") <= F.col("consumo_kwh")) & (F.col("consumo_kwh_ucs_com_gd") >= 0))
dq_check(t, "em aberto <= faturas; não verdes <= faturas", g,
         (F.col("qtd_faturas_em_aberto") <= F.col("qtd_faturas")) & (F.col("qtd_faturas_bandeira_nao_verde") <= F.col("qtd_faturas")))
dq_check(t, "valor em aberto <= receita total; receita total >= receita de energia", g,
         (F.col("valor_em_aberto") <= F.col("receita_total")) & (F.col("receita_total") >= F.col("receita_energia")))
dq_check(t, "segmento sem fatura: somas zeradas", g.filter("qtd_faturas = 0"),
         (F.col("consumo_kwh") == 0) & (F.col("receita_energia") == 0) & (F.col("qtd_ucs_faturadas") == 0))

# Reconciliação com a fato (somas exatas)
s_g = g.agg(*[F.sum(c).alias(c) for c in ["consumo_kwh", "consumo_kwh_ucs_com_gd", "receita_energia", "receita_total", "qtd_faturas",
                                          "qtd_faturas_em_aberto", "valor_em_aberto", "qtd_faturas_bandeira_nao_verde",
                                          "qtd_trocas_medidor", "qtd_ucs_com_gd"]]).first()
s_f = fato.agg(F.sum("consumo_kwh").alias("consumo_kwh"),
               F.sum(F.when(F.col("possui_gd_no_mes"), F.col("consumo_kwh"))).alias("consumo_kwh_ucs_com_gd"),
               F.sum("valor_energia").alias("receita_energia"), F.sum("valor_total").alias("receita_total"),
               F.count("*").alias("qtd_faturas"), F.sum(F.col("em_aberto").cast("int")).alias("qtd_faturas_em_aberto"),
               F.sum(F.when(F.col("em_aberto"), F.col("valor_total"))).alias("valor_em_aberto"),
               F.sum(F.when(F.col("bandeira_tarifaria") != "Verde", 1).otherwise(0)).alias("qtd_faturas_bandeira_nao_verde"),
               F.sum(F.col("troca_medidor").cast("int")).alias("qtd_trocas_medidor"),
               F.sum(F.col("possui_gd_no_mes").cast("int")).alias("qtd_ucs_com_gd")).first()
for c in ["receita_energia", "receita_total", "valor_em_aberto"]:
    dq_check_valor(t, f"soma de {c} = fato", n_g, s_g[c], s_f[c])
for c in ["consumo_kwh", "consumo_kwh_ucs_com_gd"]:
    dq_check_valor(t, f"soma de {c} = fato", n_g, s_g[c], Decimal(s_f[c]).quantize(Decimal("0.01")))
for c in ["qtd_faturas", "qtd_faturas_em_aberto", "qtd_faturas_bandeira_nao_verde", "qtd_trocas_medidor", "qtd_ucs_com_gd"]:
    dq_check_valor(t, f"soma de {c} = fato", n_g, s_g[c], s_f[c])

# Números do diagnóstico e do case
per = {r["p"]: r for r in g.withColumn("p", F.when(F.col("mes_referencia").between(P0_INI, P0_FIM), "p0")
                                            .when(F.col("mes_referencia").between(P1_INI, P1_FIM), "p1"))
       .filter("p IS NOT NULL").groupBy("p").agg(F.sum("consumo_kwh").alias("kwh"), F.sum("receita_energia").alias("rec")).collect()}
dq_check_valor(t, "volume do primeiro ano = 227.052.927 kWh", n_g, per["p0"]["kwh"], Decimal("227052927.00"))
dq_check_valor(t, "volume dos últimos 12 meses = 218.079.054 kWh", n_g, per["p1"]["kwh"], Decimal("218079054.00"))
dq_check_valor(t, "receita do primeiro ano = R$ 117.830.394,38", n_g, per["p0"]["rec"], Decimal("117830394.38"))
dq_check_valor(t, "receita dos últimos 12 meses = R$ 127.262.595,65", n_g, per["p1"]["rec"], Decimal("127262595.65"))
dq_check_valor(t, "UCs com GD em 2026-06 = 3.510", n_g, g.filter(f"mes_referencia = '{MES_FIM}'").agg(F.sum("qtd_ucs_com_gd")).first()[0], 3_510)
dq_check_valor(t, "trocas de medidor = 6.299", n_g, s_g["qtd_trocas_medidor"], 6_299)
dq_check_valor(t, "16 colunas, como no contrato", n_g, len(df_seg.columns), 16)

## 8. Registro das checagens e gravação, só se todas passarem

O log vai **sempre** para `silver.dq_log` (`table_name = gold.fato_segmento_mensal`); a tabela só é gravada se **todas** as checagens passarem.

In [0]:
spark.createDataFrame(dq_results).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tabela_dq_log)

falhas = [r for r in dq_results if not r.passed]
if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_gold} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_gold} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

df_seg.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_gold)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_gold} gravada: {n_br(spark.table(tabela_gold).count())} linhas")

## 9. Conferências com as outras Golds e respostas às perguntas

1. **Com a `kpis_mensal`** (contrato: "`qtd_ucs_com_gd` é igual à soma da A3 no mês"): a soma dos segmentos de cada mês tem de ser igual à linha do mês na `kpis_mensal`, em todas as colunas em comum.
2. **Com a `receita_decomposicao`:** somando os 12 meses de cada período por segmento, volume e receita têm de ser iguais aos da A4, linha a linha.
3. **A função de decomposição da API**, com as fórmulas da A4, para dois períodos quaisquer e qualquer filtro.
4. **As perguntas do Henrique (P10):** "Qual bairro mais perdeu volume residencial no último ano?" e "Quanto a receita comercial cresceu em Jaboatão?".

In [0]:
s = spark.table(tabela_gold)

# 1. kpis_mensal: a soma dos segmentos de cada mês = a linha do mês (esperado: 0 meses diferentes)
if spark.catalog.tableExists(tabela_kpis):
    comuns = ["qtd_ucs_cadastradas", "qtd_ucs_faturadas", "qtd_ucs_com_gd", "consumo_total_kwh", "receita_energia",
              "receita_total", "qtd_faturas_em_aberto"]
    mes_seg = s.groupBy("mes_referencia").agg(
        F.sum("qtd_ucs_cadastradas").alias("qtd_ucs_cadastradas"), F.sum("qtd_ucs_faturadas").alias("qtd_ucs_faturadas"),
        F.sum("qtd_ucs_com_gd").alias("qtd_ucs_com_gd"), F.sum("consumo_kwh").alias("consumo_total_kwh"),
        F.sum("receita_energia").alias("receita_energia"), F.sum("receita_total").alias("receita_total"),
        F.sum("qtd_faturas_em_aberto").alias("qtd_faturas_em_aberto"))
    k = spark.table(tabela_kpis).select("mes_referencia", *[F.col(c).alias(f"k_{c}") for c in comuns])
    cmp = mes_seg.join(k, "mes_referencia", "full")
    for c in comuns:
        print(f"meses com {c} diferente da kpis_mensal:", cmp.filter(~F.col(c).eqNullSafe(F.col(f"k_{c}"))).count())
else:
    print(f"{tabela_kpis} não existe neste workspace: conferência com a kpis_mensal fica para quem tiver as duas")

# 2. receita_decomposicao: volume e receita dos dois períodos, por segmento (esperado: 0 segmentos diferentes)
if spark.catalog.tableExists(tabela_rec):
    seg_p = (s.withColumn("p", F.when(F.col("mes_referencia").between(P0_INI, P0_FIM), 0)
                                .when(F.col("mes_referencia").between(P1_INI, P1_FIM), 1))
             .filter("p IS NOT NULL AND qtd_faturas > 0").groupBy("municipio", "bairro", "classe")   # a A4 só tem segmentos com fatura
             .agg(F.sum(F.when(F.col("p") == 0, F.col("consumo_kwh")).otherwise(0)).alias("v0"),
                  F.sum(F.when(F.col("p") == 1, F.col("consumo_kwh")).otherwise(0)).alias("v1"),
                  F.sum(F.when(F.col("p") == 0, F.col("receita_energia")).otherwise(0)).alias("r0"),
                  F.sum(F.when(F.col("p") == 1, F.col("receita_energia")).otherwise(0)).alias("r1")))
    rec = spark.table(tabela_rec).select("municipio", "bairro", "classe", "volume_kwh_0", "volume_kwh_1", "receita_0", "receita_1")
    c2 = seg_p.join(rec, ["municipio", "bairro", "classe"], "full")
    print("segmentos com volume ou receita diferente da receita_decomposicao:",
          c2.filter(~(F.col("v0").eqNullSafe(F.col("volume_kwh_0")) & F.col("v1").eqNullSafe(F.col("volume_kwh_1"))
                      & F.col("r0").eqNullSafe(F.col("receita_0")) & F.col("r1").eqNullSafe(F.col("receita_1")))).count())
else:
    print(f"{tabela_rec} não existe neste workspace: conferência com a receita_decomposicao fica para quem tiver as duas")

In [0]:
# 3. A decomposição da API: dois períodos quaisquer, qualquer filtro, fórmulas da A4
def decompor(ini0, fim0, ini1, fim1, filtro="true"):
    """Soma volume e receita de cada período (com o filtro) e aplica as fórmulas da A4, inclusive a regra de volume zero."""
    r = (s.filter(filtro)
         .agg(F.sum(F.when(F.col("mes_referencia").between(ini0, fim0), F.col("consumo_kwh"))).alias("v0"),
              F.sum(F.when(F.col("mes_referencia").between(ini1, fim1), F.col("consumo_kwh"))).alias("v1"),
              F.sum(F.when(F.col("mes_referencia").between(ini0, fim0), F.col("receita_energia"))).alias("r0"),
              F.sum(F.when(F.col("mes_referencia").between(ini1, fim1), F.col("receita_energia"))).alias("r1"))).first()
    v0, v1, r0, r1 = [float(r[c] or 0) for c in ("v0", "v1", "r0", "r1")]
    var = r1 - r0
    ef_vol = (v1 - v0) * (r0 / v0) if (v0 > 0 and v1 > 0) else var
    return {"volume_0": v0, "volume_1": v1, "receita_0": round(r0, 2), "receita_1": round(r1, 2),
            "variacao_volume_pct": round((v1 / v0 - 1) * 100, 2) if v0 else None,
            "variacao_receita": round(var, 2), "variacao_receita_pct": round(var / r0 * 100, 2) if r0 else None,
            "efeito_volume": round(ef_vol, 2), "efeito_preco": round(var - ef_vol, 2)}

# Confere com o diagnóstico (esperado: efeito-volume −4.657.042 e efeito-preço +14.089.243)
print("empresa, primeiro ano x últimos 12 meses:", decompor(P0_INI, P0_FIM, P1_INI, P1_FIM))

In [0]:
# 4a. "Qual bairro mais perdeu volume residencial no último ano?" (últimos 12 meses x os 12 anteriores)
ano = (s.filter("classe = 'RESIDENCIAL'")
       .groupBy("municipio", "bairro")
       .agg(F.sum(F.when(F.col("mes_referencia").between(PA_INI, PA_FIM), F.col("consumo_kwh"))).alias("volume_ano_anterior"),
            F.sum(F.when(F.col("mes_referencia").between(P1_INI, P1_FIM), F.col("consumo_kwh"))).alias("volume_ultimo_ano"),
            F.max(F.when(F.col("mes_referencia") == P1_FIM, F.col("qtd_ucs_com_gd"))).alias("ucs_com_gd_2026_06"))
       .withColumn("perda_kwh", F.col("volume_ultimo_ano") - F.col("volume_ano_anterior"))
       .withColumn("variacao_pct", F.round((F.col("volume_ultimo_ano") / F.col("volume_ano_anterior") - 1) * 100, 1)))
print("Bairros que mais perderam volume residencial no último ano (2025-07 a 2026-06 x 2024-07 a 2025-06):")
display(ano.orderBy("perda_kwh").limit(10))

# 4b. "Quanto a receita comercial cresceu em Jaboatão?" (último ano x ano anterior; e x primeiro ano)
filtro_jab = "municipio = 'Jaboatão dos Guararapes' AND classe = 'COMERCIAL'"
print("Jaboatão, comercial, último ano x ano anterior:", decompor(PA_INI, PA_FIM, P1_INI, P1_FIM, filtro_jab))
print("Jaboatão, comercial, últimos 12 meses x primeiro ano:", decompor(P0_INI, P0_FIM, P1_INI, P1_FIM, filtro_jab))

# Tamanho da tabela por mês (para o Dev dimensionar a API)
display(s.groupBy("mes_referencia").count().agg(F.min("count").alias("min_linhas_mes"), F.max("count").alias("max_linhas_mes")))